# Barranquilla MOPAC: ¿desde qué tamaño de región QM conviene MOPAC en la GPU en QM/MM?

En las sesiones QM/MM (OpenMM + MOPAC), las regiones QM de **100 átomos o menos** (contando los átomos de enlace)
corren MOPAC en la **CPU** por defecto (`src/interface/mopac_api_operations.F90`); OpenMM y la malla PME usan la GPU.
El umbral se fijó en una T4 con solo dos tamaños medidos (16 átomos: CPU más rápida; 138: GPU más rápida).
`MOPAC_FORCEGPU=1` lleva MOPAC a la GPU también por debajo del umbral.

La celda 3 aplica una corrección de la API de MOPAC (también presente en MOPAC 23.1.2): el valor de MS se calculaba
sin la carga, y toda región QM de carga impar se detenía. Varias regiones del barrido tienen carga ±1.

Este notebook mide, en la misma máquina y con el mismo binario:

| Bloque | Qué mide | Tiempo (A100) |
|---|---|---|
| A. Tripsina–BPTI | sitio activo (54 átomos QM + 5 de enlace, ~46 900 átomos, QM-PME): MOPAC en CPU frente a GPU; NVE de 400 pasos en ambos | ~15 min |
| B. Paso de 1 fs | tripsina–BPTI con `--dt 1.0`: tiempo y NVE, CPU y GPU | ~10 min |
| C. Barrido de tamaño | chignolina con los residuos 1..k en la región QM (k = 1, 2, 3, 4, 6, 8, 10), en agua: CPU frente a GPU | ~20 min |

Todo con 0,5 fs (salvo el bloque B), enlaces con H restringidos, Langevin 300 K, PM7/MOZYME, semilla fija. La CPU es la de la misma máquina de Colab (todos sus
núcleos, como la ruta por defecto).

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU A100*. Los resultados se guardan en
Google Drive (`MyDrive/mopac_paper_bench/RESULTS_NAME`); si la sesión se corta, ejecute de nuevo todas las celdas y
las corridas ya hechas se leen sin repetirse. La última celda descarga un zip.

## 1. Configuración

In [ ]:
COMMIT = 'fb16b813'            # v1.1.1-gpu + la corrección de MS (fuera de ella, el QM/MM es el de los benchmarks, c85eb96d)
RESULTS_NAME = 'qmmm_gpu_threshold_a100'
USE_DRIVE = True
SEED = 11                      # semilla de Langevin: la misma trayectoria en CPU y GPU hasta donde lo permita la aritmética
ENZYME_STEPS, ENZYME_NVE = 400, 400
SCAN_STEPS = 300
SCAN_K = [1, 2, 3, 4, 6, 8, 10]   # residuos 1..k de la chignolina (GYDPETGTWG) en la región QM
QUICK = False                  # True: 20 pasos y dos tamaños (prueba del notebook, ~15 min con la compilación)

## 2. GPU, CPU y carpeta de resultados

In [ ]:
import os, re, subprocess, sys, time, shutil, json
from pathlib import Path

def run(cmd, check=True, env=None, log=None):
    # Run a shell command and stream its output to the cell (and to a log file)
    print('$', cmd, flush=True)
    full = dict(os.environ); full.update(env or {})
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=full)
    fh = open(log, 'w') if log else None
    for line in p.stdout:
        print(line, end='', flush=True)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if check and p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

run('nvidia-smi')
gpu = subprocess.run('nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip()
cpu = subprocess.run("lscpu | grep 'Model name' | sed 's/.*: *//'", shell=True, capture_output=True,
                     text=True).stdout.strip()
NCPU = os.cpu_count()
print(f'GPU: {gpu}\nCPU: {cpu}, {NCPU} núcleos lógicos')
CC = gpu.split(',')[-1].strip().replace('.', '')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/mopac_paper_bench') / RESULTS_NAME
else:
    OUT = Path('/content/mopac_paper_bench') / RESULTS_NAME
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'env.json').write_text(json.dumps({'gpu': gpu, 'cpu': cpu, 'ncpu': NCPU, 'commit': COMMIT}, indent=1))
print('resultados en', OUT)

## 3. Compilar MOPAC (biblioteca) con OpenBLAS

~10 minutos.

In [ ]:
SRC, BUILD = Path('/content/mopac_gpu'), Path('/content/build')
run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq gfortran ninja-build '
    'libopenblas-dev liblapack-dev > /dev/null')
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
shutil.rmtree(SRC, ignore_errors=True); shutil.rmtree(BUILD, ignore_errors=True)
run(f'git clone -q https://github.com/juvenalyosa/mopac_gpu.git {SRC}')
run(f'git -C {SRC} checkout -q {COMMIT}')
run(f'git -C {SRC} log -1 --format="%h %ad %s" --date=short')
head = subprocess.run(f'git -C {SRC} rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
if not head.startswith(COMMIT):
    raise SystemExit(f'HEAD {head} no es COMMIT {COMMIT}')
stamp = OUT / 'commit.txt'
if stamp.exists() and not head.startswith(stamp.read_text().strip()[:8]):
    raise SystemExit(f'{OUT} tiene resultados de {stamp.read_text().strip()}; use otro RESULTS_NAME')
stamp.write_text(head + '\n')
# corrección de la API (MS calculado sin la carga: toda región QM de carga impar se detenía con
# "Value of MS not consistent with number of electrons"); se aplica si el commit aún no la tiene
api = SRC / 'src' / 'interface' / 'mopac_api_initialize.F90'
t = api.read_text()
old = '    nelectron = sum(tore(nat(:numat)))\n'
if old in t:
    api.write_text(t.replace(old, '    nelectron = nint(sum(tore(nat(:numat)))) - system%charge\n'))
    print('parche de MS aplicado a', api)
(OUT / 'patch.txt').write_text('MS parity with the charge: ' + ('applied' if old in t else 'already in the code') + '\n')
t0 = time.time()
run(f'cmake -S {SRC} -B {BUILD} -G Ninja -DGPU=ON -DTESTS=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_ARCHS={CC} '
    f'-DENABLE_MKL=OFF -DBLA_VENDOR=OpenBLAS > /content/cmake.log 2>&1 || (tail -40 /content/cmake.log; exit 1)')
run(f'cmake --build {BUILD} --target mopac --parallel {NCPU} > /content/build.log 2>&1 '
    f'|| (grep -i -B2 -A6 error /content/build.log | head -80; exit 1)')
LIBMOPAC = next(BUILD.glob('libmopac.so*'))
print(f'compilado en {time.time() - t0:.0f} s: {LIBMOPAC}')

## 4. OpenMM y CuPy

In [ ]:
m = re.search(r'release (\d+)', subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout)
cupy_pkg = f'cupy-cuda{m.group(1) if m else 12}x'
other = [p for p in subprocess.run('pip list 2>/dev/null', shell=True, capture_output=True, text=True).stdout.split()
         if p.startswith('cupy-cuda') and p != cupy_pkg]
if other:
    run('pip uninstall -y -q ' + ' '.join(other), check=False)
run(f"pip install -q 'openmm[cuda12]==8.6.1' {cupy_pkg} 2>&1 | tail -2", check=False)
run(f"{sys.executable} -c \"import openmm as m; print('OpenMM', m.__version__, "
    f"[m.Platform.getPlatform(i).getName() for i in range(m.Platform.getNumPlatforms())])\"", check=False)
run(f"cd {SRC}/scripts/qmmm && {sys.executable} -c \"from qmmm_pme import gpu_available; "
    f"print('QM-PME grid on the GPU (CuPy):', gpu_available())\"", check=False)

## 5. Lanzador

Cada corrida es `scripts/qmmm/run_qmmm_md.py` en su propia carpeta. Modos:

- `cpu`: `MOPAC_NOGPU=1` (MOPAC en la CPU con todos los núcleos; OpenMM y PME en la GPU). Es lo que hace hoy la ruta
  por defecto con 100 átomos QM o menos.
- `gpu`: `MOPAC_FORCEGPU=1` (MOPAC/MOZYME en la GPU aunque la región sea pequeña).

Para comprobar que la GPU se usó de verdad, se cuentan en la salida de MOPAC (`MOPAC_API_OUTPUT`) las líneas
`[MOZYME GPU` que escribe `MOPAC_GPU_DEBUG=1`.

In [ ]:
RX = {
    'system': re.compile(r'system: (\d+) atoms, (\d+) residues, box ([0-9.]+) A'),
    'qm': re.compile(r'QM region: (\d+) atoms \+ (\d+) link atoms, charge (-?\d+); long-range QM-MM electrostatics: (.+)'),
    'nvt': re.compile(r'QM/MM NVT: (\d+) steps x ([0-9.]+) fs in ([0-9.]+) s = ([0-9.]+) s/step, ([0-9.]+) ps/day'),
    'nve': re.compile(r'NVE check (\d+) steps: drift (-?[0-9.]+) kJ/mol, slope of a linear fit (-?[0-9.]+) kJ/mol/ps, '
                      r'largest step ([0-9.]+), rms step ([0-9.]+), rms deviation from the fit ([0-9.]+)'),
}
QMMM = SRC / 'scripts' / 'qmmm' / 'run_qmmm_md.py'

def qmmm(exp, case, mode, pdb, opts, steps, nve=0, mm_ps=10.0):
    d = OUT / 'runs' / exp / case / mode
    cmd = (f'{sys.executable} -u {QMMM} {pdb} --libmopac {LIBMOPAC} {opts} --platform CUDA '
           f'--steps {steps} --nve {nve} --mm-equil-ps {mm_ps} --seed {SEED}')
    if (d / 'result.json').exists() and (d / 'cmd.txt').exists() and (d / 'cmd.txt').read_text() == cmd:
        r = json.loads((d / 'result.json').read_text())
        print(f'[{exp}/{case}] {mode}: ya hecho, {r.get("s_per_step")} s/step')
        return r
    shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True)
    (d / 'cmd.txt').write_text(cmd)
    api_out = Path('/tmp') / f'mopac_api_{exp}_{case}_{mode}.out'
    api_out.unlink(missing_ok=True)
    env = {k: v for k, v in os.environ.items() if k not in ('MOPAC_NOGPU', 'MOPAC_FORCEGPU')}
    env.update(OMP_NUM_THREADS=str(NCPU), OPENBLAS_NUM_THREADS=str(NCPU), MOPAC_GPU_DEBUG='1',
               MOPAC_API_OUTPUT=str(api_out))
    env['MOPAC_NOGPU' if mode == 'cpu' else 'MOPAC_FORCEGPU'] = '1'
    print(f'[{exp}/{case}] {mode}', flush=True)
    t0 = time.perf_counter()
    with open(d / 'log.txt', 'w') as fh:
        rc = subprocess.run(cmd, shell=True, cwd=d, env=env, stdout=fh, stderr=subprocess.STDOUT).returncode
    text = (d / 'log.txt').read_text(errors='ignore')
    r = {'exp': exp, 'case': case, 'mode': mode, 'rc': rc, 'total_s': round(time.perf_counter() - t0, 1)}
    if (m := RX['system'].search(text)):
        r.update(total_atoms=int(m.group(1)), box_A=float(m.group(3)))
    if (m := RX['qm'].search(text)):
        r.update(qm_atoms=int(m.group(1)), link_atoms=int(m.group(2)), qm_charge=int(m.group(3)), long_range=m.group(4).strip())
    if (m := RX['nvt'].search(text)):
        r.update(steps=int(m.group(1)), dt_fs=float(m.group(2)), md_s=float(m.group(3)),
                 s_per_step=float(m.group(4)), ps_per_day=float(m.group(5)))
    if (m := RX['nve'].search(text)):
        r.update(nve_steps=int(m.group(1)), nve_drift_kj=float(m.group(2)), nve_slope_kj_ps=float(m.group(3)),
                 nve_rms_dev_kj=float(m.group(6)))
    # evidence of where MOPAC ran
    lines = api_out.read_text(errors='ignore').splitlines() if api_out.exists() else []
    gl = [l for l in lines if '[MOZYME GPU' in l]
    r['mozyme_gpu_lines'] = len(gl)
    (d / 'gpu_lines.txt').write_text('\n'.join(gl[:60] + ['...'] + gl[-20:]) + '\n')
    if rc != 0:
        (d / 'mopac_api_tail.out').write_text('\n'.join(lines[-400:]) + '\n')
    api_out.unlink(missing_ok=True)
    (d / ('result.json' if rc == 0 and 's_per_step' in r else 'result_failed.json')).write_text(json.dumps(r, indent=1))
    print(f'    {r.get("s_per_step")} s/step, {r.get("ps_per_day")} ps/day, QM {r.get("qm_atoms")}+{r.get("link_atoms")} '
          f'atoms, {r.get("total_atoms")} in all, [MOZYME GPU] lines {len(gl)}, rc {rc}, {r["total_s"]:.0f} s', flush=True)
    if rc != 0:
        print('\n'.join(text.splitlines()[-10:]))
    return r

def get_pdb(code):
    import urllib.request
    f = OUT / 'runs' / '_pdb' / f'{code}.pdb'
    f.parent.mkdir(parents=True, exist_ok=True)
    if not f.exists():
        urllib.request.urlretrieve(f'https://files.rcsb.org/download/{code}.pdb', f)
    return f

## A. Tripsina–BPTI (2PTC): sitio activo con QM-PME

Las mismas opciones que la corrida del artículo (`exp_qmmm_enzyme`, caso `pme`, sin la prueba de cut-off):
Ser195, His57 (HID) y Asp102 (cadenas laterales) de la tripsina y Lys15–Ala16 del BPTI; 100 ps de equilibrado MM.

In [ ]:
ENZ = ('--qm-residues E:57:sc,E:102:sc,E:195:sc,I:15,I:16 --variants E:57=HID --padding 1.0 --ron 10 --roff 12 '
       '--distances E:195:OG-I:15:C,E:57:NE2-E:195:OG,E:57:HD1-E:102:OD2')
pdb = get_pdb('2PTC')
enz = {}
for mode in ('cpu', 'gpu'):
    enz[mode] = qmmm('enzyme', 'trypsin_pme', mode, pdb, ENZ, 20 if QUICK else ENZYME_STEPS,
                     0 if QUICK else ENZYME_NVE, mm_ps=10.0 if QUICK else 100.0)

## B. Tripsina–BPTI con pasos de 1 fs

El sistema QM/MM ya restringe los enlaces con hidrógeno (`constraints=HBonds` en `run_qmmm_md.py`). ¿Se conserva
la energía con 1 fs? Las mismas corridas del bloque A con `--dt 1.0` y la prueba NVE de 400 pasos (0,4 ps).

In [ ]:
for mode in ('cpu', 'gpu'):
    enz[mode + '_1fs'] = qmmm('enzyme', 'trypsin_pme_1fs', mode, pdb, ENZ + ' --dt 1.0', 20 if QUICK else ENZYME_STEPS,
                              0 if QUICK else ENZYME_NVE, mm_ps=10.0 if QUICK else 100.0)

## C. Barrido del tamaño de la región QM (chignolina en agua)

Residuos 1..k de la chignolina (1UAO optimizada) como región QM, con átomos de enlace donde se corta la cadena; la
misma caja de agua (padding 2.5 nm, ron 6, roff 8 Å, QM-PME) que la corrida del artículo de la chignolina completa
(k = 10, 138 átomos). Solo cambia el tamaño de la región QM.

In [ ]:
CHIG = SRC / 'tests' / 'data' / 'mozyme_gpu_reference' / '1UAO_opt.pdb'
scan = []
for k in ([2, 10] if QUICK else SCAN_K):
    opts = f'--qm-residues {",".join(map(str, range(1, k + 1)))} --padding 2.5 --ron 6 --roff 8'
    for mode in ('cpu', 'gpu'):
        r = qmmm('scan', f'chig_1-{k}', mode, CHIG, opts, 20 if QUICK else SCAN_STEPS)
        scan.append(dict(r, k=k))

## 6. Resultados

In [ ]:
import pandas as pd
rows = []
for f in sorted((OUT / 'runs').rglob('result*.json')):
    r = json.loads(f.read_text())
    r['ok'] = f.name == 'result.json'
    rows.append(r)
df = pd.DataFrame(rows)
df['qm_total'] = df['qm_atoms'] + df['link_atoms']
df.to_csv(OUT / 'qmmm_threshold.csv', index=False)
cols = ['exp', 'case', 'mode', 'qm_atoms', 'link_atoms', 'qm_total', 'qm_charge', 'total_atoms', 'steps', 'dt_fs', 's_per_step', 'ps_per_day',
        'mozyme_gpu_lines', 'nve_drift_kj', 'nve_rms_dev_kj', 'ok']
display(df[[c for c in cols if c in df]].sort_values(['exp', 'qm_total', 'mode']))

piv = df[df['ok']].pivot_table(index=['exp', 'case', 'qm_total', 'dt_fs'], columns='mode', values='s_per_step')
if {'cpu', 'gpu'} <= set(piv.columns):
    piv['GPU más rápida en'] = piv['cpu'] / piv['gpu']
    display(piv.sort_index(level='qm_total'))
    piv.to_csv(OUT / 'qmmm_threshold_pivot.csv')

import matplotlib.pyplot as plt
s = df[(df['exp'] == 'scan') & df['ok']]
fig, ax = plt.subplots(figsize=(5, 3.4))
for mode, color, mk in (('cpu', '#eb6834', 'o'), ('gpu', '#2a78d6', 's')):
    g = s[s['mode'] == mode].sort_values('qm_total')
    ax.plot(g['qm_total'], g['s_per_step'], marker=mk, color=color, label=f'MOPAC en la {mode.upper()}')
e = df[(df['exp'] == 'enzyme') & df['ok'] & (df['dt_fs'] == 0.5)]
for mode, color in (('cpu', '#eb6834'), ('gpu', '#2a78d6')):
    g = e[e['mode'] == mode]
    if len(g):
        ax.plot(g['qm_total'], g['s_per_step'], ls='none', marker='D', mfc='white', color=color,
                label=f'tripsina–BPTI, {mode.upper()}')
ax.axvline(100.5, color='gray', lw=0.8, ls=':')
ax.set(xlabel='átomos QM (con átomos de enlace)', ylabel='s por paso (0,5 fs)', yscale='log')
ax.legend(fontsize=7, frameon=False)
fig.tight_layout()
fig.savefig(OUT / 'qmmm_threshold.png', dpi=200)
plt.show()

## 7. Descargar

In [ ]:
tmp = Path('/content/_qmmm_thr') / RESULTS_NAME
shutil.rmtree(tmp.parent, ignore_errors=True)
tmp.mkdir(parents=True)
for f in ('qmmm_threshold.csv', 'qmmm_threshold_pivot.csv', 'qmmm_threshold.png', 'env.json', 'commit.txt', 'patch.txt'):
    if (OUT / f).exists():
        shutil.copy2(OUT / f, tmp / f)
for f in list((OUT / 'runs').rglob('*.json')) + list((OUT / 'runs').rglob('log.txt')) + \
        list((OUT / 'runs').rglob('gpu_lines.txt')) + list((OUT / 'runs').rglob('cmd.txt')) + \
        list((OUT / 'runs').rglob('mopac_api_tail.out')):
    dst = tmp / 'runs' / f.relative_to(OUT / 'runs')
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(f, dst)
z = shutil.make_archive(f'/content/{RESULTS_NAME}', 'zip', tmp.parent)
print(z)
from google.colab import files
files.download(z)